### Import libraries

This cell imports pandas for reading and working with tabular data, and NumPy for numerical operations and missing-value handling.

In [36]:
import re

import pandas as pd
import numpy as np

## 1. Imports and configuration

This cell documents the notebook configuration. The paths identify the four raw datasets, and the sentinel values identify codes that should become missing values.

In [37]:
from pathlib import Path

DATA_DIR = Path('../data')
MISSING_VALUES = [-666666666, -6666666666]
ZIP_PATTERN = r'\d{5}'

DATASETS = {
    'public_features': DATA_DIR / 'miami_dade_public_features.csv',
    'area_features': DATA_DIR / 'area_features.csv',
    'area_options': DATA_DIR / 'area_options.csv',
    'crowd_text_snippets': DATA_DIR / 'crowd_text_snippets.csv',
}

## 2. Reusable functions

These functions keep the cleaning rules consistent across datasets. ZIP codes are handled as strings, missing sentinel values are converted to `NaN`, summary values are extracted only from their labels, and categorical fields are one-hot encoded.

In [48]:
'''
load_dataset()- load all 4 dataset
'''
def load_dataset(path):
    """Load a CSV while preserving ZIP codes as strings."""
    return pd.read_csv(path, dtype={'zip': 'string'})

'''
replace_missing_values()- replace all the missing values( -6666666)  into NaN across 4 dataset
'''
def replace_missing_values(df, missing_values=MISSING_VALUES):
    """Replace configured sentinel values with pandas missing values."""
    return df.replace(missing_values, np.nan)


def extract_migration_score(text):
    """Extract a labeled migration score, returning 0 when unavailable."""
    if pd.isna(text):
        return 0.0

    match = re.search(
        r'migration\s+score\s*[:=]?\s*(-?\d+(?:\.\d+)?)',
        str(text),
        flags=re.IGNORECASE,
    )
    if match is None:
        return 0.0

    try:
        return round(float(match.group(1)), 2)
    except ValueError:
        return 0.0


def extract_median_rent(text):
    """Extract a labeled median rent as an integer, returning 0 when unavailable."""
    if pd.isna(text):
        return 0

    match = re.search(
        r'median\s+rent\s*[:=]?\s*~?\s*\$?\s*([\d,]+(?:\.\d+)?)',
        str(text),
        flags=re.IGNORECASE,
    )
    if match is None:
        return 0

    try:
        return int(float(match.group(1).replace(',', '')))
    except ValueError:
        return 0


def add_summary_numeric_columns(df, summary_column='summary'):
    """Add summary-derived numeric columns without overwriting existing columns."""
    result = df.copy()
    audit = {
        'migration_score_missing_or_invalid': 0,
        'median_rent_missing_or_invalid': 0,
        'migration_score_existing': 'migration_score' in result.columns,
        'median_rent_existing': 'median_rent' in result.columns,
    }

    if summary_column not in result.columns:
        raise KeyError(f'Missing required summary column: {summary_column}')

    if 'migration_score' not in result.columns:
        migration_values = result[summary_column].map(extract_migration_score)
        audit['migration_score_missing_or_invalid'] = int((migration_values == 0).sum())
        result['migration_score'] = migration_values.astype(float).round(2)

    if 'median_rent' not in result.columns:
        rent_values = result[summary_column].map(extract_median_rent)
        audit['median_rent_missing_or_invalid'] = int((rent_values == 0).sum())
        result['median_rent'] = rent_values.astype(int)

    return result, audit

'''
drop_columns_by_dataset()- drop dataset-specific columns.

'\miami_dade_public_features.csv'
NAME

'\area_features.csv'
area_name

'\area_options.csv'
Disclaimer

'\crowd_text_snippets.csv'
Source
'''
def drop_columns_by_dataset(datasets, columns_to_drop):
    """Drop configured columns using case-insensitive name matching."""
    cleaned_datasets = {}
    dropped_columns = {}
    missing_columns = {}

    for name, dataset in datasets.items():
        requested_columns = columns_to_drop.get(name, [])
        column_lookup = {column.casefold(): column for column in dataset.columns}
        existing_columns = [
            column_lookup[column.casefold()]
            for column in requested_columns
            if column.casefold() in column_lookup
        ]
        missing_columns[name] = [
            column for column in requested_columns
            if column.casefold() not in column_lookup
        ]
        cleaned_datasets[name] = dataset.drop(columns=existing_columns)
        dropped_columns[name] = existing_columns

    return cleaned_datasets, dropped_columns, missing_columns

'''
drop_rows_with_excessive_missing_values()- Drop row with more than 3 NaN 
'''
def drop_rows_with_excessive_missing_values(df, max_missing=3):
    """Drop rows with more than max_missing NaN values."""
    missing_counts = df.isna().sum(axis=1)
    keep_rows = missing_counts <= max_missing
    return df.loc[keep_rows].copy(), missing_counts

'''
validate_zip_codes()- ensure zip code are all 5 digit and is not empty, else drop the row
'''
def validate_zip_codes(df, zip_column='zip'):
    """Keep rows whose ZIP value is present and exactly five digits."""
    cleaned_df = df.copy()
    zip_values = cleaned_df[zip_column].astype('string').str.strip()
    valid_zip = zip_values.notna() & zip_values.str.fullmatch(ZIP_PATTERN).fillna(False)
    cleaned_df[zip_column] = zip_values
    return cleaned_df.loc[valid_zip].copy()

'''
one_hot_encode()- one hot encode string columns
rent_band
housing_preference
household
theme

'''
def one_hot_encode(df, columns):
    """One-hot encode selected categorical columns and remove their originals."""
    existing_columns = [column for column in columns if column in df.columns]
    return pd.get_dummies(df, columns=existing_columns, dtype=int)

'''
one_hot_encode_tags()- one hot encode string columns with tags
tags
'''
def one_hot_encode_tags(df, column='tags'):
    """Split comma-separated tags and create one binary column per tag."""
    if column not in df.columns:
        return df.copy()

    tags = df[column].fillna('').astype('string').str.get_dummies(sep=',')
    tags.columns = [f'tag_{tag.strip()}' for tag in tags.columns]
    tags = tags.loc[:, tags.columns != 'tag_']
    return pd.concat([df.drop(columns=[column]), tags], axis=1)

<>:79: SyntaxWarning: invalid escape sequence '\m'
<>:79: SyntaxWarning: invalid escape sequence '\m'
/var/folders/zt/0c46hq3s317cq_09dx9btg600000gn/T/ipykernel_279/2595231773.py:79: SyntaxWarning: invalid escape sequence '\m'
  '''


## 3. Load all four datasets

This cell loads the public features, area features, area options, and crowd text datasets. The reusable loader preserves the `zip` column as text so leading zeroes are not lost.

In [39]:
datasets = {
    name: load_dataset(path)
    for name, path in DATASETS.items()
}

public_features = datasets['public_features']
area_features = datasets['area_features']
area_options = datasets['area_options']
crowd_text_snippets = datasets['crowd_text_snippets']

## 4. Display original shapes and schemas

This cell records each dataset's size, column names, and data types before cleaning. This gives us a baseline for comparing the cleaned outputs.

In [28]:
original_shapes = {}

for name, dataset in datasets.items():
    original_shapes[name] = dataset.shape
    print(f'\n{name}: shape={dataset.shape}')
    print('columns:', dataset.columns.tolist())
    print(dataset.dtypes)


public_features: shape=(80, 17)
columns: ['NAME', 'B01003_001E', 'B25064_001E', 'B25077_001E', 'B07003_001E', 'B07003_004E', 'B07003_008E', 'B07003_010E', 'B11016_001E', 'B25003_002E', 'B25003_003E', 'zip', 'moved_different_house_pct', 'median_rent_usd', 'population', 'zillow_home_value_index', 'zillow_rent_index']
NAME                                 object
B01003_001E                           int64
B25064_001E                           int64
B25077_001E                           int64
B07003_001E                           int64
B07003_004E                           int64
B07003_008E                           int64
B07003_010E                           int64
B11016_001E                           int64
B25003_002E                           int64
B25003_003E                           int64
zip                          string[python]
moved_different_house_pct           float64
median_rent_usd                       int64
population                            int64
zillow_home_value_inde

## 5. Replace missing sentinel values

This cell applies the same reusable missing-value function to all four datasets. Both sentinel spellings are handled, and the replacement counts are printed for auditing.

In [40]:
sentinel_counts = {}

for name, dataset in datasets.items():
    sentinel_mask = dataset.isin(MISSING_VALUES)
    sentinel_counts[name] = int(sentinel_mask.to_numpy().sum())
    datasets[name] = replace_missing_values(dataset)
    print(f'{name}: replaced {sentinel_counts[name]} sentinel values')

public_features = datasets['public_features']
area_features = datasets['area_features']
area_options = datasets['area_options']
crowd_text_snippets = datasets['crowd_text_snippets']

public_features: replaced 8 sentinel values
area_features: replaced 3 sentinel values
area_options: replaced 0 sentinel values
crowd_text_snippets: replaced 0 sentinel values


## 6. Drop dataset-specific columns

This cell removes columns that are not needed for the cleaned datasets. Edge Case: Missing requested columns are reported instead of causing an error.

In [49]:
columns_to_drop = {
    'public_features': ['NAME'],
    'area_features': ['area_name'],
    'area_options': ['Disclaimer','option_id','summary'],
    'crowd_text_snippets': ['Source','snippet_id'],
}

datasets, dropped_columns, missing_columns = drop_columns_by_dataset(
    datasets,
    columns_to_drop,
)

for name in datasets:
    print(f'{name}: dropped columns={dropped_columns[name]}')
    if missing_columns[name]:
        print(f'{name}: requested columns not found={missing_columns[name]}')

public_features = datasets['public_features']
area_features = datasets['area_features']
area_options = datasets['area_options']
crowd_text_snippets = datasets['crowd_text_snippets']

public_features: dropped columns=[]
public_features: requested columns not found=['NAME']
area_features: dropped columns=[]
area_features: requested columns not found=['area_name']
area_options: dropped columns=['summary']
area_options: requested columns not found=['Disclaimer', 'option_id']
crowd_text_snippets: dropped columns=[]
crowd_text_snippets: requested columns not found=['Source', 'snippet_id']


## 7. Extract migration score and median rent ( from area_options.csv)

This cell reads labeled values from `area_options.summary`. Missing or malformed values become numeric zero, and existing target columns are preserved rather than overwritten. Extraction counts are printed so zero values caused by missing summaries remain auditable.

In [42]:
area_options, summary_extraction_audit = add_summary_numeric_columns(
    area_options,
    summary_column='summary',
)

datasets['area_options'] = area_options

print('Summary extraction audit:')
print(summary_extraction_audit)
print(area_options[['summary', 'migration_score', 'median_rent']].head())

Summary extraction audit:
{'migration_score_missing_or_invalid': 6, 'median_rent_missing_or_invalid': 0, 'migration_score_existing': False, 'median_rent_existing': False}
                                             summary  migration_score  \
0  Wynwood (ZIP 33127) — your own apartment (e.g....              0.0   
1  Downtown Miami (ZIP 33128) — your own apartmen...              0.0   
2  Brickell (ZIP 33130) — your own apartment with...              0.0   
3  Wynwood (ZIP 33127) — your own studio or 1BR, ...              0.0   
4  Downtown Miami (ZIP 33128) — co-living option ...              0.0   

   median_rent  
0         1283  
1         1191  
2         1553  
3         1283  
4         1191  


## 8. Drop rows with too many missing values

Rows with more than three `NaN` values are removed. This rule is applied after summary extraction, and only retained data fields count toward the threshold. Rows with exactly three missing values remain.

In [43]:
missing_row_counts = {}

for name, dataset in datasets.items():
    cleaned_dataset, missing_counts = drop_rows_with_excessive_missing_values(
        dataset,
        max_missing=3,
    )
    missing_row_counts[name] = {
        'before': len(dataset),
        'after': len(cleaned_dataset),
        'dropped': len(dataset) - len(cleaned_dataset),
    }
    datasets[name] = cleaned_dataset
    print(f'{name}: dropped {missing_row_counts[name]["dropped"]} rows')

public_features = datasets['public_features']
area_features = datasets['area_features']
area_options = datasets['area_options']
crowd_text_snippets = datasets['crowd_text_snippets']

public_features: dropped 2 rows
area_features: dropped 0 rows
area_options: dropped 0 rows
crowd_text_snippets: dropped 0 rows


## 9. Validate ZIP columns

This cell applies the reusable ZIP validation function to every dataset after column, summary, and missing-row cleaning. Rows with an empty, missing, or non-five-digit ZIP are removed and counted.

In [44]:
zip_validation_counts = {}

for name, dataset in datasets.items():
    validated_dataset = validate_zip_codes(dataset)
    zip_validation_counts[name] = {
        'before': len(dataset),
        'after': len(validated_dataset),
        'dropped': len(dataset) - len(validated_dataset),
    }
    datasets[name] = validated_dataset
    print(f'{name}: kept {len(validated_dataset)} of {len(dataset)} rows')

public_features = datasets['public_features']
area_features = datasets['area_features']
area_options = datasets['area_options']
crowd_text_snippets = datasets['crowd_text_snippets']

public_features: kept 78 of 78 rows
area_features: kept 80 of 80 rows
area_options: kept 10 of 10 rows
crowd_text_snippets: kept 20 of 20 rows


## 10. Apply one-hot encoding

This cell encodes categorical columns while leaving identifiers, ZIP codes, numeric summary values, and free text unchanged. Tags are split into separate binary columns because one row can contain multiple tags.

In [45]:
area_features = one_hot_encode(area_features, ['rent_band'])
area_options = one_hot_encode(area_options, ['rent_band', 'housing_preference', 'household'])
area_options = one_hot_encode_tags(area_options, 'tags')
crowd_text_snippets = one_hot_encode(crowd_text_snippets, ['theme'])

datasets.update({
    'public_features': public_features,
    'area_features': area_features,
    'area_options': area_options,
    'crowd_text_snippets': crowd_text_snippets,
})

for name, dataset in datasets.items():
    print(f'\n{name} after one-hot encoding:')
    print(dataset.head())
    print('columns:', dataset.columns.tolist())


public_features after one-hot encoding:
   B01003_001E  B25064_001E  B25077_001E  B07003_001E  B07003_004E  \
1          792          NaN    2000001.0          792          630   
2         1873       2370.0    1335900.0         1824          962   
3        54873       1238.0     339300.0        54290        49530   
4        46798       1546.0     287100.0        46403        41635   
5        28306       1283.0     346000.0        27993        24314   

   B07003_008E  B07003_010E  B11016_001E  B25003_002E  B25003_003E    zip  \
1           97            0          328          307           21  33109   
2          241            8          816           76          740  33122   
3         2057          171        21227         5080        16147  33125   
4         1763          240        18073         7104        10969  33126   
5         1533          300         9661         2819         6842  33127   

   moved_different_house_pct  median_rent_usd  population  \
1             

## 11. Print cleaned results

This cell prints a preview and the shape of every cleaned dataset so we can inspect the transformations, including the extracted summary values.

In [50]:
for name, dataset in datasets.items():
    print(f'\n{name}: shape={dataset.shape}')
    display(dataset.head(10))


public_features: shape=(78, 16)


,B01003_001E,B25064_001E,B25077_001E,B07003_001E,B07003_004E,B07003_008E,B07003_010E,B11016_001E,B25003_002E,B25003_003E,zip,moved_different_house_pct,median_rent_usd,population,zillow_home_value_index,zillow_rent_index
1,792,NaN,2000001.0,792,630,97,0,328,307,21,33109,0.204545,NaN,792,6.413596e+06,NaN
2,1873,2370.0,1335900.0,1824,962,241,8,816,76,740,33122,0.472588,2370.0,1873,1.505538e+06,2740.611111
3,54873,1238.0,339300.0,54290,49530,2057,171,21227,5080,16147,33125,0.087677,1238.0,54873,4.597721e+05,2313.579167
4,46798,1546.0,287100.0,46403,41635,1763,240,18073,7104,10969,33126,0.102752,1546.0,46798,3.017829e+05,2380.241026
5,28306,1283.0,346000.0,27993,24314,1533,300,9661,2819,6842,33127,0.131426,1283.0,28306,5.213359e+05,2998.258333
6,8989,1191.0,292400.0,8937,7310,753,60,3907,123,3784,33128,0.182052,1191.0,8989,4.407685e+05,2809.281197
7,15184,2153.0,526800.0,14954,12414,912,76,7267,3393,3874,33129,0.169854,2153.0,15184,6.876460e+05,3644.846979
8,32946,1553.0,436000.0,32573,25611,2841,620,16915,2120,14795,33130,0.213735,1553.0,32946,5.072061e+05,3150.938320
9,22540,2649.0,599100.0,22256,15720,1931,418,12552,4132,8420,33131,0.293674,2649.0,22540,5.887397e+05,3793.482203
10,17136,2373.0,535900.0,16658,10852,1936,1075,8969,2226,6743,33132,0.348541,2373.0,17136,5.297232e+05,3286.435059



area_features: shape=(80, 15)


,zip,transit,social,quiet,pet_friendly,walkable,migration_score,median_rent_usd,co_living_friendly,single_family_friendly,rent_band_1200-1800,rent_band_1800-2200,rent_band_2200-2800,rent_band_2800-3500,rent_band_3500+
0,33101,0.000000,0.000000,1.000000,0.6,0.000000,0.000000,NaN,0.150000,0.650000,1,0,0,0,0
1,33109,0.000000,0.011141,0.994429,0.6,0.000000,0.432820,NaN,0.179813,0.650000,1,0,0,0,0
2,33122,0.461831,0.026348,0.986826,0.6,0.999998,1.000000,2370.0,0.415809,0.650000,0,0,1,0,0
3,33125,0.883777,0.771902,0.614049,0.6,0.999997,0.185526,1238.0,0.537352,0.495454,1,0,0,0,0
4,33126,0.707822,0.658311,0.670845,0.6,0.999997,0.217424,1546.0,0.481956,0.557929,1,0,0,0,0
5,33127,0.852804,0.800000,0.800909,0.6,0.999997,0.278098,1283.0,0.920000,0.580454,1,0,0,0,0
6,33128,0.918624,0.800000,0.936776,0.6,0.999997,0.385224,1191.0,0.880000,0.648388,1,0,0,0,0
7,33129,0.508357,0.213594,0.893203,0.6,0.999998,0.359413,2153.0,0.780000,0.650000,0,1,0,0,0
8,33130,0.850000,0.800000,0.768273,0.6,0.999997,0.452266,1553.0,0.900000,0.564137,1,0,0,0,0
9,33131,0.850000,0.800000,0.841464,0.6,0.999999,0.621416,2649.0,0.550000,0.600732,0,0,1,0,0



area_options: shape=(10, 16)


,zip,migration_score,median_rent,rent_band_1200-1800,rent_band_1800-2200,rent_band_2200-2800,housing_preference_co_living,housing_preference_own_apartment,household_alone,household_with_co_leaser,tag_co_living,tag_pet_friendly,tag_quiet,tag_social,tag_transit,tag_walkable
0,33127,0.00,1283,1,0,0,0,1,0,1,0,0,0,1,1,1
1,33128,0.00,1191,1,0,0,0,1,0,1,0,0,0,0,1,1
2,33130,0.00,1553,1,0,0,0,1,0,1,0,0,0,1,1,1
3,33127,0.00,1283,1,0,0,0,1,1,0,0,0,0,1,1,1
4,33128,0.00,1191,1,0,0,1,0,1,0,1,0,0,1,1,0
5,33130,0.00,1553,1,0,0,1,0,1,0,1,0,0,1,1,0
6,33132,0.74,2373,0,0,1,0,1,1,0,0,0,0,1,1,1
7,33139,0.53,1680,1,0,0,0,1,1,0,0,0,0,1,1,1
8,33137,0.48,2142,0,1,0,0,1,1,0,0,0,0,1,1,1
9,33133,0.41,1824,0,1,0,0,1,0,1,0,1,1,0,0,0



crowd_text_snippets: shape=(20, 9)


,zip,text,theme_co_living,theme_expensive,theme_pet_friendly,theme_quiet,theme_single_family,theme_social,theme_transit
0,33127,Great walkability and easy bus access; felt we...,0,0,0,0,0,0,1
1,33127,Lots of events and people meeting up; can be l...,0,0,0,0,0,1,0
2,33130,Metrorail nearby made commuting easy without a...,0,0,0,0,0,0,1
3,33130,Busy area; street noise is real in some towers.,0,0,0,1,0,0,0
4,33139,Very social neighborhood; beach culture and ni...,0,0,0,0,0,1,0
5,33139,Rent is high but you pay for location and energy.,0,1,0,0,0,0,0
6,33133,Tree-lined streets and parks; quieter than dow...,0,0,0,1,0,0,0
7,33133,Saw many dog owners; parks are a plus.,0,0,1,0,0,0,0
8,33137,Creative scene; good for meeting people after ...,0,0,0,0,0,1,0
9,33142,Affordable compared to Brickell; research bloc...,0,0,0,1,0,0,0


## 12. Check data types across all four datasets

This cell displays the data type of every column after cleaning. It helps confirm that ZIP codes remain strings, numeric columns are numeric, and one-hot encoded columns are integers.

In [51]:
for name, dataset in datasets.items():
    print(f'\n{name} data types:')
    display(dataset.dtypes.to_frame(name='data_type'))


public_features data types:


,data_type
B01003_001E,int64
B25064_001E,float64
B25077_001E,float64
B07003_001E,int64
B07003_004E,int64
B07003_008E,int64
B07003_010E,int64
B11016_001E,int64
B25003_002E,int64
B25003_003E,int64



area_features data types:


,data_type
zip,string[python]
transit,float64
social,float64
quiet,float64
pet_friendly,float64
walkable,float64
migration_score,float64
median_rent_usd,float64
co_living_friendly,float64
single_family_friendly,float64



area_options data types:


,data_type
zip,string[python]
migration_score,float64
median_rent,int64
rent_band_1200-1800,int64
rent_band_1800-2200,int64
rent_band_2200-2800,int64
housing_preference_co_living,int64
housing_preference_own_apartment,int64
household_alone,int64
household_with_co_leaser,int64



crowd_text_snippets data types:


,data_type
zip,string[python]
text,object
theme_co_living,int64
theme_expensive,int64
theme_pet_friendly,int64
theme_quiet,int64
theme_single_family,int64
theme_social,int64
theme_transit,int64


# Data Quality check

check all dataset are cleaned

In [46]:
for name, dataset in datasets.items():
    remaining_sentinels = int(dataset.isin(MISSING_VALUES).to_numpy().sum())
    valid_zip = dataset['zip'].notna() & dataset['zip'].str.fullmatch(ZIP_PATTERN).fillna(False)
    duplicate_columns = dataset.columns[dataset.columns.duplicated()].tolist()

    assert remaining_sentinels == 0, f'{name} still contains sentinel values'
    assert valid_zip.all(), f'{name} contains an invalid ZIP code'
    assert not duplicate_columns, f'{name} contains duplicate columns: {duplicate_columns}'

    print(f'{name}: validation passed; rows={len(dataset)}, columns={len(dataset.columns)}')

assert pd.api.types.is_float_dtype(area_options['migration_score'])
assert pd.api.types.is_integer_dtype(area_options['median_rent'])
assert area_options['migration_score'].between(0, 1).all()
assert (area_options['median_rent'] >= 0).all()
print('area_options summary values: validation passed')

public_features: validation passed; rows=78, columns=16
area_features: validation passed; rows=80, columns=15
area_options: validation passed; rows=10, columns=17
crowd_text_snippets: validation passed; rows=20, columns=9
area_options summary values: validation passed


## 12. Run final validation checks

This cell verifies that no configured sentinel values remain, all ZIP values are valid five-digit strings, summary-derived values are numeric and valid, and the cleaned datasets have no duplicate column names.

## 12. Save cleaned datasets

This cell writes each cleaned DataFrame to a new CSV with `_cleaned` in the filename. The original raw datasets are not overwritten.

In [ ]:
# output_paths = {}

# for name, dataset in datasets.items():
#     output_path = DATA_DIR / f'{name}_cleaned.csv'
#     dataset.to_csv(output_path, index=False)
#     output_paths[name] = output_path
#     print(f'Saved {name} to {output_path}')

Saved public_features to ../data/public_features_cleaned.csv
Saved area_features to ../data/area_features_cleaned.csv
Saved area_options to ../data/area_options_cleaned.csv
Saved crowd_text_snippets to ../data/crowd_text_snippets_cleaned.csv
